# Predictive-coding pretraining: behaviour analysis

Compare three regimes on a downstream action-recognition task:

1. **PT only** &mdash; CTM core after Kinetics predictive-coding pretraining (no finetuning). What does the predictor head + sync representation already encode?
2. **PT + FT** &mdash; pretrained CTM core then finetuned on the downstream task.
3. **Baseline (FT only)** &mdash; identical recipe but with random-init CTM core.

Sections:
- Setup &mdash; point the three `CKPT` variables at your runs.
- A. Pretraining sanity &mdash; predictor cosine vs. trivial baselines.
- B. Downstream accuracy &mdash; per-tick / most-certain accuracy, PT+FT vs Baseline.
- C. Sync evolution &mdash; how does `synch_out` move across ticks before and after each regime?
- D. Attention patterns &mdash; what are the three regimes attending to?

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print('cwd:', os.getcwd())

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

from tasks.pretrain.model import CTMVideoPredictiveCoding
from tasks.pretrain.dataset import build_finetune_datasets, build_pretrain_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

# Edit these paths once your runs have produced checkpoints.
CKPT_PRETRAIN = 'logs/pretrain/kinetics/checkpoint.pt'
CKPT_FINETUNE = 'logs/pretrain/finetune_ucf101/checkpoint.pt'
CKPT_BASELINE = 'logs/pretrain/baseline_ucf101/checkpoint.pt'

# Pick the eval dataset to match the finetuning task above.
EVAL_DATASET = 'ucf101'                     # 'ucf101' | 'hmdb51' | 'synthetic'
EVAL_DATA_ROOT = '/geovic/geovic/UCF-101'   # edit me

In [ ]:
# Helper: rebuild a model from a saved args namespace and load its core weights.

def build_model_from_args(args, n_classes, device=DEVICE):
    model = CTMVideoPredictiveCoding(
        n_frames=args.n_frames,
        iterations_per_frame=args.iterations_per_frame,
        d_model=args.d_model,
        d_input=args.d_input,
        heads=args.heads,
        n_synch_out=args.n_synch_out,
        n_synch_action=args.n_synch_action,
        synapse_depth=args.synapse_depth,
        memory_length=args.memory_length,
        deep_nlms=args.deep_memory,
        memory_hidden_dims=args.memory_hidden_dims,
        do_layernorm_nlm=args.do_normalisation,
        backbone_type=args.backbone_type,
        positional_embedding_type=args.positional_embedding_type,
        out_dims=n_classes,
        prediction_reshaper=[-1],
        dropout=args.dropout,
        dropout_nlm=args.dropout_nlm,
        neuron_select_type=args.neuron_select_type,
    ).to(device)
    dummy = torch.randn(1, args.n_frames, 3, args.image_size, args.image_size, device=device)
    with torch.no_grad():
        _ = model(dummy)
    return model


def load_finetuned(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = ckpt['args']
    class_labels = ckpt['class_labels']
    model = build_model_from_args(args, len(class_labels))
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    model.eval()
    return model, ckpt


def load_pretrained(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = ckpt['args']
    # We need a number of classes for the output_projector; pick the eval dataset's count.
    _, test_data, class_labels = build_finetune_datasets(
        EVAL_DATASET, EVAL_DATA_ROOT, args.n_frames, args.image_size, fold=getattr(args, 'fold', 1)
    )
    model = build_model_from_args(args, len(class_labels))
    model.load_core_state_dict(ckpt.get('core_state_dict') or ckpt['model_state_dict'],
                                drop_predictor_head=False, drop_output_projector=True)
    model.eval()
    return model, ckpt, test_data, class_labels

## A. Pretraining sanity

How well does the predictor anticipate next-frame features? Compare against two trivial baselines:
- **identity**: predict frame_{f+1} == frame_f (i.e., zero-motion). Cosine sim of frame_f vs frame_{f+1}.
- **shuffle**: cosine sim of frame_f vs a random other frame (within the same clip, but not f+1).

In [ ]:
model_pt, ckpt_pt, test_data, class_labels = load_pretrained(CKPT_PRETRAIN)
args = ckpt_pt['args']
loader = torch.utils.data.DataLoader(test_data, batch_size=8, shuffle=True, num_workers=0)

predictor_cos, identity_cos, shuffle_cos = [], [], []
with torch.no_grad():
    for i, (clips, _) in enumerate(loader):
        clips = clips.to(DEVICE)
        loss, cos_pred = model_pt.predictive_coding_loss(clips)  # cos_pred: (B, n_frames-1)

        # Identity & shuffle baselines on the kv features themselves.
        kv = model_pt.compute_features(clips)              # (B, T, N, d_input)
        feat = kv.mean(dim=2)                              # (B, T, d_input)
        feat_n = F.normalize(feat, dim=-1)
        # Identity: cosine(feat_t, feat_{t+1}).
        id_cos = (feat_n[:, :-1] * feat_n[:, 1:]).sum(-1)  # (B, T-1)
        # Shuffle: cosine(feat_t, feat_{shuffled}) where shuffled != t+1.
        T = feat.shape[1]
        shuf = []
        for t in range(T - 1):
            forbidden = {t, t + 1}
            choices = [k for k in range(T) if k not in forbidden]
            shuf.append(np.random.choice(choices))
        shuf = torch.tensor(shuf, device=DEVICE)
        sh_cos = (feat_n[:, :-1] * feat_n[:, shuf]).sum(-1)

        predictor_cos.append(cos_pred.cpu().numpy())
        identity_cos.append(id_cos.cpu().numpy())
        shuffle_cos.append(sh_cos.cpu().numpy())
        if i >= 20: break

predictor_cos = np.concatenate(predictor_cos)
identity_cos = np.concatenate(identity_cos)
shuffle_cos = np.concatenate(shuffle_cos)

print(f'predictor cos:  mean={predictor_cos.mean():.3f}  std={predictor_cos.std():.3f}')
print(f'identity cos:   mean={identity_cos.mean():.3f}  std={identity_cos.std():.3f}')
print(f'shuffle cos:    mean={shuffle_cos.mean():.3f}  std={shuffle_cos.std():.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([predictor_cos.flatten(), identity_cos.flatten(), shuffle_cos.flatten()],
        bins=40, label=['predictor', 'identity (frame_t≈frame_{t+1})', 'shuffle'],
        alpha=0.7, density=True)
ax.legend(); ax.set_xlabel('cosine similarity'); ax.set_title('Pretraining alignment vs trivial baselines')
plt.tight_layout(); plt.show()

## B. Downstream accuracy: PT+FT vs baseline

Compare per-tick (=per-frame) accuracy and the most-certain accuracy from the saved metric histories. Higher / earlier-rising curves indicate that the predictive-coding pretraining helped.

In [ ]:
ft_ckpt = torch.load(CKPT_FINETUNE, map_location='cpu', weights_only=False)
bl_ckpt = torch.load(CKPT_BASELINE, map_location='cpu', weights_only=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ft_ckpt['iters'], ft_ckpt['train_acc_mc'], 'b-', label='PT+FT train')
axes[0].plot(ft_ckpt['iters'], ft_ckpt['test_acc_mc'], 'b--', label='PT+FT test')
axes[0].plot(bl_ckpt['iters'], bl_ckpt['train_acc_mc'], 'r-', label='baseline train')
axes[0].plot(bl_ckpt['iters'], bl_ckpt['test_acc_mc'], 'r--', label='baseline test')
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('most-certain accuracy')
axes[0].legend(); axes[0].set_title('Most-certain accuracy')

ft_pt = np.array(ft_ckpt['test_acc_per_tick'])[-1]
bl_pt = np.array(bl_ckpt['test_acc_per_tick'])[-1]
axes[1].plot(np.arange(len(ft_pt)), ft_pt, 'b-o', label='PT+FT')
axes[1].plot(np.arange(len(bl_pt)), bl_pt, 'r-o', label='baseline')
axes[1].set_xlabel('tick (=frame)'); axes[1].set_ylabel('test accuracy')
axes[1].legend(); axes[1].set_title('Per-tick test accuracy at end of training')
plt.tight_layout(); plt.show()

print(f'PT+FT  final test most-certain: {ft_ckpt["test_acc_mc"][-1]:.3f}')
print(f'Baseline  final test most-certain: {bl_ckpt["test_acc_mc"][-1]:.3f}')

## C. Sync trajectory across ticks (3 regimes side by side)

Run a single clip through each model with `track=True` and compare how `synch_out` drifts across ticks. PCA the (n_ticks, n_synch_out) trajectory to 2D.

In [ ]:
from sklearn.decomposition import PCA

model_ft, _ = load_finetuned(CKPT_FINETUNE)
model_bl, _ = load_finetuned(CKPT_BASELINE)
# model_pt is the pre-train-only model loaded earlier.

clip, label = test_data[np.random.randint(len(test_data))]
x = clip.unsqueeze(0).to(DEVICE)

def get_sync_trajectory(model, x):
    out = model(x, track=True)
    sync_out_track = out[2][0]  # shape (T_ticks, B, n_synch_out)
    return sync_out_track[:, 0]  # (T_ticks, n_synch_out)

with torch.no_grad():
    traj_pt = get_sync_trajectory(model_pt, x)
    traj_ft = get_sync_trajectory(model_ft, x)
    traj_bl = get_sync_trajectory(model_bl, x)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, traj, name in zip(axes, [traj_pt, traj_ft, traj_bl], ['PT only', 'PT+FT', 'baseline']):
    p = PCA(n_components=2).fit_transform(traj)
    ticks = np.arange(p.shape[0])
    sc = ax.scatter(p[:, 0], p[:, 1], c=ticks, cmap='viridis', s=30)
    ax.plot(p[:, 0], p[:, 1], '-', color='gray', alpha=0.5)
    ax.set_title(f'{name}  (label={class_labels[label]})')
    plt.colorbar(sc, ax=ax, label='tick')
plt.tight_layout(); plt.show()

## D. Attention heatmaps per regime

For one frame mid-clip, average the attention map over heads and overlay on the frame for each regime.

In [ ]:
def get_attention(model, x):
    out = model(x, track=True)
    attn = out[5]  # (T_ticks, B, heads, 1, n_tokens) numpy
    spatial_shape = out[7]
    return attn[:, 0].mean(axis=1).squeeze(1), spatial_shape  # (T_ticks, n_tokens)

with torch.no_grad():
    attn_pt, sp = get_attention(model_pt, x)
    attn_ft, _ = get_attention(model_ft, x)
    attn_bl, _ = get_attention(model_bl, x)

T_total = attn_pt.shape[0]
n_ticks_per_frame = max(1, getattr(args, 'iterations_per_frame', 1))
show_frame = T_total // 2
Hp, Wp = sp

frame = clip[show_frame].cpu().numpy().transpose(1, 2, 0)
frame = (frame - frame.min()) / (frame.max() - frame.min() + 1e-8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(frame); axes[0].set_title(f'frame {show_frame}'); axes[0].axis('off')
for ax, attn, name in zip(axes[1:], [attn_pt, attn_ft, attn_bl], ['PT only', 'PT+FT', 'baseline']):
    a = attn[show_frame].reshape(Hp, Wp)
    ax.imshow(frame); ax.imshow(a, cmap='hot', alpha=0.5,
                                 extent=(0, frame.shape[1], frame.shape[0], 0))
    ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()